# **CNN-BERT (FakeBERT)**

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # "0" o "1"

In [2]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus[0], 'GPU')
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print("Using GPU:", gpus[0])
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found, using CPU.")

2025-11-07 20:39:19.839063: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-07 20:39:19.894540: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-07 20:39:21.093556: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Using GPU: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [3]:
from utils import *

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from transformers import BertTokenizer, TFBertModel

/home/n.emmolo/miniconda3/envs/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Ignore warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")

In [5]:
# ---------------
# BERT Embeddings
# ---------------

tokenizer = BertTokenizer.from_pretrained("bert-base-cased")
bert_model = TFBertModel.from_pretrained("bert-base-cased", from_pt=True)

def get_bert_embeddings(texts, max_len=128, batch_size=16):
    """
    Get BERT embeddings for a list of texts.

    Args:
        texts: List or array of input texts
        max_len: Maximum length for padding/truncation
        batch_size: Batch size for processing texts

    Returns:
        Numpy array of BERT embeddings with shape (num_texts, max_len, 768)
    """
    
    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size].tolist()
        input_enc = tokenizer(
            batch_texts,
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors='tf'
        )
        outputs = bert_model(input_enc)
        batch_emb = outputs.last_hidden_state  # (batch, max_len, 768)
        embeddings.append(batch_emb.numpy())

        # libera memoria GPU tra un batch e l’altro
        del input_enc, outputs, batch_emb
        tf.keras.backend.clear_session()

    return np.concatenate(embeddings, axis=0)


def generate_bert_embeddings(datasets):
    """
    Generate BERT embeddings for all datasets.
    
    Args:
        datasets: Dictionary of datasets with train/val/test splits.
    
    Returns:
        Updated datasets with BERT embeddings.
    """

    for name, data in datasets.items():
        print(f"\n=== Generating embeddings for dataset: {name} ===")

        X_train, y_train = data["train"]
        X_val, y_val = data["val"]
        X_test, y_test = data["test"]

        X_train_emb = get_bert_embeddings(X_train)
        X_val_emb = get_bert_embeddings(X_val)
        X_test_emb = get_bert_embeddings(X_test)

        datasets[name] = {
            "train": (X_train_emb, y_train),
            "val": (X_val_emb, y_val),
            "test": (X_test_emb, y_test)
        }
    
    return datasets

I0000 00:00:1762544366.443151 3965618 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 29756 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:af:00.0, compute capability: 7.0
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining 

In [6]:
# ------------------------------
# Build model function
# ------------------------------

def build_model(max_len=128, cnn_filters=96, kernel_size=4,
                dense_units=32, learning_rate=1e-4):
    """
    Builds a CNN model on top of BERT embeddings (not end-to-end fine-tuning).

    Args:
        max_len (int): Maximum sequence length.
        cnn_filters (int): Number of filters in Conv1D layer.
        kernel_size (int): Size of convolution kernel.
        dense_units (int): Units in dense hidden layer.
        learning_rate (float): Learning rate for Adam optimizer.

    Returns:
        model (tf.keras.Model): Compiled CNN-BERT model.
    """
    model = Sequential([
        Input(shape=(max_len, 768)),  # BERT base hidden size
        Conv1D(filters=cnn_filters, kernel_size=kernel_size, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(dense_units, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

## VERSION 1: Dataset (Simple)

In [ ]:
dataset_df = data_loading() # load datasets

for name, df in dataset_df.items():
    print(f"Dataset: {name}, Number of samples: {len(df)}")

print("\nSplitting datasets into train/val/test...")
datasets = {name: split_dataset(df) for name, df in dataset_df.items()} # split all datasets in train/val/test
print("\nComputing BERT embeddings for all datasets...")
datasets = generate_bert_embeddings(datasets) # get BERT embeddings for all datasets

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


Dataset: Celebrity, Number of samples: 500
Dataset: CIDII, Number of samples: 722
Dataset: FaKES, Number of samples: 842
Dataset: FakeVsSatire, Number of samples: 486
Dataset: Horne, Number of samples: 326
Dataset: Infodemic, Number of samples: 10559
Dataset: ISOT, Number of samples: 44271
Dataset: Kaggle_clement, Number of samples: 39105
Dataset: Kaggle_meg, Number of samples: 12845
Dataset: LIAR_PLUS, Number of samples: 12784
Dataset: Politifact, Number of samples: 504
Dataset: Unipi_NDF, Number of samples: 554

Splitting datasets into train/val/test...

Computing BERT embeddings for all datasets...

=== Generating embeddings for dataset: Celebrity ===

=== Generating embeddings for dataset: CIDII ===

=== Generating embeddings for dataset: FaKES ===

=== Generating embeddings for dataset: FakeVsSatire ===

=== Generating embeddings for dataset: Horne ===

=== Generating embeddings for dataset: Infodemic ===

=== Generating embeddings for dataset: ISOT ===

=== Generating embeddings 

In [ ]:
# --------------------------------
# Fine-tuning on multiple datasets
# --------------------------------

model = build_model() # initialize model

results = {}

# sequential training
for i, (name, data) in enumerate(datasets.items()):
    print(f"\n=== Phase {i+1}: Training/Fine-tuning on {name} ===")
    
    X_train, y_train = data["train"]
    X_val, y_val = data["val"]
    X_test, y_test = data["test"]

    # early stopping
    es = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True, verbose=0)

    # fine-tune on train + val
    model.fit(
        np.concatenate([X_train, X_val]),
        np.concatenate([y_train, y_val]),
        epochs=10,
        batch_size=16,
        validation_split=0.1,
        callbacks=[es],
        verbose=1
    )

    y_pred = model.predict(X_test)
    y_pred = (y_pred > 0.5).astype(int)
    print(f"Classification Report after {name}:")
    print(classification_report(y_test, y_pred))
    print(f"Confusion Matrix after {name}:")
    print(confusion_matrix(y_test, y_pred))
    print(f"\nWeighted F1-score after {name}:", f1_score(y_test, y_pred, average="weighted"))


    # evaluation on all datasets
    print("\n--- Evaluation on all datasets ---")
    results[name] = {}
    for test_name, test_data in datasets.items(): # for each dataset
        X_te, y_te = test_data["test"]
        preds = model.predict(X_te)
        preds = (preds > 0.5).astype(int)
        f1 = f1_score(y_te, preds, average="weighted")
        results[name][test_name] = f1
        print(f"Evaluation on {test_name}: Weighted F1 = {f1:.4f}")



=== Phase 1: Training/Fine-tuning on Celebrity ===
Epoch 1/10


2025-11-07 13:14:25.396951: I external/local_xla/xla/service/service.cc:163] XLA service 0x7d6004015b70 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-11-07 13:14:25.396990: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Tesla V100S-PCIE-32GB, Compute Capability 7.0
2025-11-07 13:14:25.451151: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-11-07 13:14:25.603201: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002


16/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4531 - loss: 0.9698

I0000 00:00:1762517666.743815 3721953 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 99ms/step - accuracy: 0.4806 - loss: 0.8098 - val_accuracy: 0.5250 - val_loss: 0.6890
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6333 - loss: 0.6324 - val_accuracy: 0.5750 - val_loss: 0.6764
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7278 - loss: 0.5772 - val_accuracy: 0.6750 - val_loss: 0.6282
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8306 - loss: 0.4999 - val_accuracy: 0.7000 - val_loss: 0.6268
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8833 - loss: 0.4506 - val_accuracy: 0.7250 - val_loss: 0.6048
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9250 - loss: 0.3839 - val_accuracy: 0.6750 - val_loss: 0.5976
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9278 - loss: 0.3604 - val_accuracy: 0.7000 - val_loss: 0.5884
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9639 - loss: 0.2980 - val_accuracy: 0.7500 - val_loss: 0.

In [ ]:
# ---------------
# Results summary
# ---------------

print("\n=== Results Summary ===")
for name, res in results.items():
    print(f"\nResults after training on {name}:")
    for test_name, f1 in res.items():
        print(f"  Test on {test_name}: Weighted F1 = {f1:.4f}")


=== Results Summary ===

Results after training on Celebrity:
  Test on Celebrity: Weighted F1 = 0.6599
  Test on CIDII: Weighted F1 = 0.6616
  Test on FaKES: Weighted F1 = 0.4752
  Test on FakeVsSatire: Weighted F1 = 0.5736
  Test on Horne: Weighted F1 = 0.4032
  Test on Infodemic: Weighted F1 = 0.3376
  Test on ISOT: Weighted F1 = 0.5530
  Test on Kaggle_clement: Weighted F1 = 0.7458
  Test on Kaggle_meg: Weighted F1 = 0.5449
  Test on LIAR_PLUS: Weighted F1 = 0.5124
  Test on Politifact: Weighted F1 = 0.5533
  Test on Unipi_NDF: Weighted F1 = 0.3618

Results after training on CIDII:
  Test on Celebrity: Weighted F1 = 0.4544
  Test on CIDII: Weighted F1 = 0.9381
  Test on FaKES: Weighted F1 = 0.3171
  Test on FakeVsSatire: Weighted F1 = 0.4278
  Test on Horne: Weighted F1 = 0.2631
  Test on Infodemic: Weighted F1 = 0.3095
  Test on ISOT: Weighted F1 = 0.3696
  Test on Kaggle_clement: Weighted F1 = 0.3137
  Test on Kaggle_meg: Weighted F1 = 0.0613
  Test on LIAR_PLUS: Weighted F1 = 0

## VERSION 2: Dataset by Topic

In [7]:
dataset_df = data_by_topic()

# split "politics" in "politics1" and "politics2"
if "politics" in dataset_df:
    dataset_df["politics1"] = dataset_df["politics"].sample(frac=0.5, random_state=42)
    dataset_df["politics2"] = dataset_df["politics"].drop(dataset_df["politics1"].index)
    # drop original "politics" dataset
    del dataset_df["politics"]
    # put "politics1" and "politics2" at the beginning of the dict
    dataset_df = {"politics1": dataset_df["politics1"], "politics2": dataset_df["politics2"], **dataset_df}

for topic, df in dataset_df.items():
    print(f"Topic: {topic}, Number of samples: {len(df)}")

#dataset_df = dict(list(dataset_df.items())[3:]) # pop first 3 datasets to reduce computation time

print("\nSplitting datasets into train/val/test...")
datasets = {topic: split_dataset(df) for topic, df in dataset_df.items()} # split all datasets in train/val/test
print("\nComputing BERT embeddings for all datasets...")
datasets = generate_bert_embeddings(datasets) # get BERT embeddings for all datasets

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


Topic: politics1, Number of samples: 48738
Topic: politics2, Number of samples: 48738
Topic: general, Number of samples: 12845
Topic: covid, Number of samples: 10559
Topic: syria, Number of samples: 842
Topic: islam, Number of samples: 722
Topic: notredame, Number of samples: 554
Topic: gossip, Number of samples: 500

Splitting datasets into train/val/test...

Computing BERT embeddings for all datasets...

=== Generating embeddings for dataset: politics1 ===

=== Generating embeddings for dataset: politics2 ===

=== Generating embeddings for dataset: general ===

=== Generating embeddings for dataset: covid ===

=== Generating embeddings for dataset: syria ===

=== Generating embeddings for dataset: islam ===

=== Generating embeddings for dataset: notredame ===

=== Generating embeddings for dataset: gossip ===


In [8]:
# -------------------------------
# Fine-tuning on Dataset by Topic
# -------------------------------

model = build_model() # initialize model

results = {}

# sequential training
for i, (topic, data) in enumerate(datasets.items()):
    print(f"\n=== Phase {i+1}: Training/Fine-tuning on topic: {topic} ===")

    X_train, y_train = data["train"]
    X_val, y_val = data["val"]
    X_test, y_test = data["test"]

    # early stopping
    es = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True, verbose=1)

    # fine-tune on train + val
    model.fit(
        np.concatenate([X_train, X_val]),
        np.concatenate([y_train, y_val]),
        epochs=10,
        batch_size=16,
        validation_data=(X_val, y_val),
        callbacks=[es],
        verbose=1
    )

    y_pred = model.predict(X_test)
    y_pred = (y_pred > 0.5).astype(int)
    print(f"Classification Report after topic {topic}:")
    print(classification_report(y_test, y_pred))
    print(f"Confusion Matrix after topic {topic}:")
    print(confusion_matrix(y_test, y_pred))
    print(f"\nWeighted F1-score after topic {topic}:", f1_score(y_test, y_pred, average="weighted"))


    # evaluation on all topics
    print("\n--- Evaluation on all topics ---")
    results[topic] = {}
    for test_topic, test_data in datasets.items(): # for each topic
        X_te, y_te = test_data["test"]
        preds = model.predict(X_te)
        preds = (preds > 0.5).astype(int)
        f1 = f1_score(y_te, preds, average="weighted")
        results[topic][test_topic] = f1
        print(f"Evaluation on topic {test_topic}: Weighted F1 = {f1:.4f}")


=== Phase 1: Training/Fine-tuning on topic: politics1 ===
Epoch 1/10


2025-11-07 21:41:33.857987: I external/local_xla/xla/service/service.cc:163] XLA service 0x722aa0002800 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-11-07 21:41:33.858026: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Tesla V100S-PCIE-32GB, Compute Capability 7.0
2025-11-07 21:41:33.938916: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-11-07 21:41:34.090668: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002


  21/2437 ━━━━━━━━━━━━━━━━━━━━ 19s 8ms/step - accuracy: 0.5339 - loss: 0.9932 

I0000 00:00:1762548095.387734 3965903 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2437/2437 ━━━━━━━━━━━━━━━━━━━━ 37s 14ms/step - accuracy: 0.9261 - loss: 0.1513 - val_accuracy: 0.9447 - val_loss: 0.1016
Epoch 2/10
2437/2437 ━━━━━━━━━━━━━━━━━━━━ 27s 11ms/step - accuracy: 0.9429 - loss: 0.1048 - val_accuracy: 0.9465 - val_loss: 0.0911
Epoch 3/10
2437/2437 ━━━━━━━━━━━━━━━━━━━━ 27s 11ms/step - accuracy: 0.9503 - loss: 0.0937 - val_accuracy: 0.9599 - val_loss: 0.0832
Epoch 4/10
2437/2437 ━━━━━━━━━━━━━━━━━━━━ 26s 11ms/step - accuracy: 0.9563 - loss: 0.0826 - val_accuracy: 0.9671 - val_loss: 0.0668
Epoch 5/10
2437/2437 ━━━━━━━━━━━━━━━━━━━━ 26s 11ms/step - accuracy: 0.9653 - loss: 0.0723 - val_accuracy: 0.9848 - val_loss: 0.0548
Epoch 6/10
2437/2437 ━━━━━━━━━━━━━━━━━━━━ 26s 11ms/step - accuracy: 0.9736 - loss: 0.0587 - val_accuracy: 0.9910 - val_loss: 0.0388
Epoch 7/10
2437/2437 ━━━━━━━━━━━━━━━━━━━━ 27s 11ms/step - accuracy: 0.9809 - loss: 0.0453 - val_accuracy: 0.9897 - val_loss: 0.0350
Epoch 8/10
2437/2437 ━━━━━━━━━━━━━━━━━━━━ 27s 11ms/step - accuracy: 0.9879 - loss: 0.03

528/528 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9991 - loss: 0.0049 - val_accuracy: 1.0000 - val_loss: 5.6833e-04
Epoch 9/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9989 - loss: 0.0046 - val_accuracy: 1.0000 - val_loss: 7.3816e-04
Epoch 10/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9986 - loss: 0.0054 - val_accuracy: 1.0000 - val_loss: 6.1389e-04
Epoch 10: early stopping
Restoring model weights from the end of the best epoch: 8.
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
Classification Report after topic covid:
              precision    recall  f1-score   support

         0.0       0.96      0.97      0.97      1106
         1.0       0.97      0.96      0.96      1006

    accuracy                           0.96      2112
   macro avg       0.96      0.96      0.96      2112
weighted avg       0.96      0.96      0.96      2112

Confusion Matrix after topic covid:
[[1075   31]
 [  44  962]]

Weighted F1-score after topic covid: 0.9644769077592933


In [9]:
# ---------------
# Results summary
# ---------------

print("\n=== Results Summary ===")
for topic, res in results.items():
    print(f"\nResults after training on topic {topic}:")
    for test_topic, f1 in res.items():
        print(f"  Test on topic {test_topic}: Weighted F1 = {f1:.4f}")


=== Results Summary ===

Results after training on topic politics1:
  Test on topic politics1: Weighted F1 = 0.9441
  Test on topic politics2: Weighted F1 = 0.9419
  Test on topic general: Weighted F1 = 0.0829
  Test on topic covid: Weighted F1 = 0.3887
  Test on topic syria: Weighted F1 = 0.3869
  Test on topic islam: Weighted F1 = 0.2619
  Test on topic notredame: Weighted F1 = 0.2635
  Test on topic gossip: Weighted F1 = 0.3333

Results after training on topic politics2:
  Test on topic politics1: Weighted F1 = 0.9428
  Test on topic politics2: Weighted F1 = 0.9437
  Test on topic general: Weighted F1 = 0.1356
  Test on topic covid: Weighted F1 = 0.4005
  Test on topic syria: Weighted F1 = 0.4797
  Test on topic islam: Weighted F1 = 0.3206
  Test on topic notredame: Weighted F1 = 0.2879
  Test on topic gossip: Weighted F1 = 0.3333

Results after training on topic general:
  Test on topic politics1: Weighted F1 = 0.3525
  Test on topic politics2: Weighted F1 = 0.3561
  Test on topic

## VERSION 3: Dataset by Date

In [22]:
dataset_df = data_by_date()

for date, df in dataset_df.items():
    print(f"Date: {date}, Number of samples: {len(df)}")

# dataset_df = dict(list(dataset_df.items())[:3]) # pop last 3 datasets to reduce computation time

print("\nSplitting datasets into train/val/test...")
datasets = {date: split_dataset(df) for date, df in dataset_df.items()} # split all datasets in train/val/test
print("\nComputing BERT embeddings for all datasets...")
datasets = generate_bert_embeddings(datasets) # get BERT embeddings for all datasets

Date: 2011-2013, Number of samples: 55
Date: 2014, Number of samples: 114
Date: 2015, Number of samples: 84
Date: 2016, Number of samples: 63018
Date: 2017, Number of samples: 16657
Date: 2019, Number of samples: 554
Date: 2020, Number of samples: 10559

Splitting datasets into train/val/test...

Computing BERT embeddings for all datasets...

=== Generating embeddings for dataset: 2011-2013 ===

=== Generating embeddings for dataset: 2014 ===

=== Generating embeddings for dataset: 2015 ===

=== Generating embeddings for dataset: 2016 ===

=== Generating embeddings for dataset: 2017 ===

=== Generating embeddings for dataset: 2019 ===

=== Generating embeddings for dataset: 2020 ===


In [23]:
# ------------------------------
# Fine-tuning on Dataset by Date
# ------------------------------

model = build_model() # initialize model

results = {}

# sequential training
for i, (date, data) in enumerate(datasets.items()):
    print(f"\n=== Phase {i+1}: Training/Fine-tuning on date: {date} ===")
    
    X_train, y_train = data["train"]
    X_val, y_val = data["val"]
    X_test, y_test = data["test"]

    # early stopping
    es = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True, verbose=1)

    # fine-tune on train + val
    model.fit(
        np.concatenate([X_train, X_val]),
        np.concatenate([y_train, y_val]),
        epochs=10,
        batch_size=64,
        validation_data=(X_val, y_val),
        callbacks=[es],
        verbose=1
    )

    y_pred = model.predict(X_test)
    y_pred = (y_pred > 0.5).astype(int)
    print(f"Classification Report after date {date}:")
    print(classification_report(y_test, y_pred))
    print(f"Confusion Matrix after date {date}:")
    print(confusion_matrix(y_test, y_pred))
    print(f"\nWeighted F1-score after date {date}:", f1_score(y_test, y_pred, average="weighted"))


    # evaluation on all dates
    print("\n--- Evaluation on all dates ---")
    results[date] = {}
    for test_date, test_data in datasets.items(): # for each date
        X_te, y_te = test_data["test"]
        preds = model.predict(X_te)
        preds = (preds > 0.5).astype(int)
        f1 = f1_score(y_te, preds, average="weighted")
        results[date][test_date] = f1
        print(f"Evaluation on {test_date}: Weighted F1 = {f1:.4f}")
    


=== Phase 1: Training/Fine-tuning on date: 2011-2013 ===
Epoch 1/10


2025-11-07 19:52:03.445409: I external/local_xla/xla/service/service.cc:163] XLA service 0x7d86f4017bf0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-11-07 19:52:03.445450: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Tesla V100S-PCIE-32GB, Compute Capability 7.0
2025-11-07 19:52:03.485515: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-11-07 19:52:03.635538: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4773 - loss: 0.8584

I0000 00:00:1762541525.438902 3875234 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.4773 - loss: 0.8584 - val_accuracy: 0.5455 - val_loss: 0.6594
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 229ms/step - accuracy: 0.6364 - loss: 0.7230 - val_accuracy: 0.6364 - val_loss: 0.5922
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step - accuracy: 0.5455 - loss: 0.7990 - val_accuracy: 0.8182 - val_loss: 0.5396
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 227ms/step - accuracy: 0.6591 - loss: 0.6238 - val_accuracy: 1.0000 - val_loss: 0.5034
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step - accuracy: 0.6136 - loss: 0.5810 - val_accuracy: 1.0000 - val_loss: 0.4741
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step - accuracy: 0.6818 - loss: 0.6315 - val_accuracy: 1.0000 - val_loss: 0.4419
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step - accuracy: 0.5909 - loss: 0.6380 - val_accuracy: 1.0000 - val_loss: 0.4128
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step - accuracy: 0.7500 - loss: 0.5273 - val_accuracy: 1.0000 - val_loss: 0.3883
Epoch 

...

2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step - accuracy: 0.4396 - loss: 0.8221 - val_accuracy: 0.5652 - val_loss: 0.6891
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - accuracy: 0.4725 - loss: 0.7867 - val_accuracy: 0.6522 - val_loss: 0.6581
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - accuracy: 0.4505 - loss: 0.8159 - val_accuracy: 0.8261 - val_loss: 0.6236
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step - accuracy: 0.6923 - loss: 0.6368 - val_accuracy: 0.8261 - val_loss: 0.5889
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step - accuracy: 0.6264 - loss: 0.6390 - val_accuracy: 0.8696 - val_loss: 0.5569
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step - accuracy: 0.8242 - loss: 0.5549 - val_accuracy: 0.9565 - val_loss: 0.5267
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step - accuracy: 0.8132 - loss: 0.5399 - val_accuracy: 0.9565 - val_loss: 0.4940
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step - accuracy: 0.8242 - loss: 0.5276 - val_accuracy: 1.0000 - val_loss: 0.4623
Epoch 

2025-11-07 19:53:31.309288: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:382] Garbage collection: deallocate free memory regions (i.e., allocations) so that we can re-allocate a larger region to avoid OOM due to memory fragmentation. If you see this message frequently, you are running near the threshold of the available device memory and re-allocation may incur great performance overhead. You may try smaller batch sizes to observe the performance impact. Set TF_ENABLE_GPU_GARBAGE_COLLECTION=false if you'd like to disable this feature.


Epoch 1/10
788/788 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8441 - loss: 0.3288

2025-11-07 19:54:02.722895: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 15.25GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.


788/788 ━━━━━━━━━━━━━━━━━━━━ 25s 32ms/step - accuracy: 0.9197 - loss: 0.2008 - val_accuracy: 0.9737 - val_loss: 0.0803
Epoch 2/10
788/788 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.9750 - loss: 0.0826 - val_accuracy: 0.9838 - val_loss: 0.0528
Epoch 3/10
788/788 ━━━━━━━━━━━━━━━━━━━━ 16s 20ms/step - accuracy: 0.9830 - loss: 0.0580 - val_accuracy: 0.9900 - val_loss: 0.0392
Epoch 4/10
788/788 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.9873 - loss: 0.0436 - val_accuracy: 0.9932 - val_loss: 0.0269
Epoch 5/10
788/788 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.9902 - loss: 0.0323 - val_accuracy: 0.9947 - val_loss: 0.0198
Epoch 6/10
788/788 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.9931 - loss: 0.0249 - val_accuracy: 0.9974 - val_loss: 0.0122
Epoch 7/10
788/788 ━━━━━━━━━━━━━━━━━━━━ 16s 20ms/step - accuracy: 0.9950 - loss: 0.0178 - val_accuracy: 0.9976 - val_loss: 0.0114
Epoch 8/10
788/788 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.9965 - loss: 0.0131 - val_accurac

In [24]:
# ---------------
# Results summary
# ---------------

print("\n=== Results Summary ===")
for date, res in results.items():
    print(f"\nResults after training on date {date}:")
    for test_date, f1 in res.items():
        print(f"  Test on date {test_date}: Weighted F1 = {f1:.4f}")


=== Results Summary ===

Results after training on date 2011-2013:
  Test on date 2011-2013: Weighted F1 = 0.4545
  Test on date 2014: Weighted F1 = 0.3671
  Test on date 2015: Weighted F1 = 0.4706
  Test on date 2016: Weighted F1 = 0.3690
  Test on date 2017: Weighted F1 = 0.4032
  Test on date 2019: Weighted F1 = 0.4217
  Test on date 2020: Weighted F1 = 0.4456

Results after training on date 2014:
  Test on date 2011-2013: Weighted F1 = 0.4455
  Test on date 2014: Weighted F1 = 0.3843
  Test on date 2015: Weighted F1 = 0.3393
  Test on date 2016: Weighted F1 = 0.3826
  Test on date 2017: Weighted F1 = 0.4435
  Test on date 2019: Weighted F1 = 0.4419
  Test on date 2020: Weighted F1 = 0.4600

Results after training on date 2015:
  Test on date 2011-2013: Weighted F1 = 0.3528
  Test on date 2014: Weighted F1 = 0.3890
  Test on date 2015: Weighted F1 = 0.1588
  Test on date 2016: Weighted F1 = 0.2736
  Test on date 2017: Weighted F1 = 0.1860
  Test on date 2019: Weighted F1 = 0.2809
 